In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [4]:
df = pd.read_csv("/content/cyclone_labels.csv")

In [5]:
conditions = [
    (df["WindSpeed"] < 34),
    (df["WindSpeed"] >= 34) & (df["WindSpeed"] < 64),
    (df["WindSpeed"] >= 64)
]

labels = ["Depression", "Tropical Storm", "Hurricane"]

df["Cyclone_Category"] = np.select(
    conditions,
    labels,
    default="Unknown"
)

In [6]:
X = df.drop(columns=[
    "CycloneID",
    "WindSpeed",
    "Cyclone_Category"
])

y = df["Cyclone_Category"]

In [7]:
print("Feature Names:", list(X.columns))
print("Target Classes:", y.unique())
print("Number of Samples:", X.shape[0])
print("Number of Features:", X.shape[1])

print(df.head())

print("\nClass Distribution")
print(y.value_counts())

Feature Names: ['Basin', 'Longitude', 'Latitude', 'DateTime', 'PressureDrop', 'Pressure']
Target Classes: ['Depression' 'Tropical Storm' 'Hurricane']
Number of Samples: 10538
Number of Features: 6
  Basin CycloneID  Longitude  Latitude    DateTime  WindSpeed  PressureDrop  \
0  ATLN   200301L      -66.0      31.4  2003041815       30.0           0.0   
1  ATLN   200301L      -66.3      31.9  2003041818       30.0           0.0   
2  ATLN   200301L      -66.6      32.5  2003041821       30.0           0.0   
3  ATLN   200301L      -68.6      34.5  2003041912       35.0           0.0   
4  ATLN   200301L      -68.8      34.4  2003041915       35.0           0.0   

   Pressure Cyclone_Category  
0    1008.0       Depression  
1    1007.0       Depression  
2    1007.0       Depression  
3    1006.0   Tropical Storm  
4    1006.0   Tropical Storm  

Class Distribution
Cyclone_Category
Tropical Storm    3904
Depression        3818
Hurricane         2816
Name: count, dtype: int64


In [8]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features),
        ("num", StandardScaler(), numerical_features)
    ]
)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)

Shape of X_train: (8430, 6)
Shape of X_test: (2108, 6)


In [11]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])


In [12]:
model.fit(X_train, y_train)

print("Model Trained Successfully!")


Model Trained Successfully!


In [13]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

print(results.head(10))

           Actual       Predicted
0      Depression      Depression
1      Depression      Depression
2      Depression      Depression
3  Tropical Storm  Tropical Storm
4      Depression      Depression
5  Tropical Storm  Tropical Storm
6      Depression      Depression
7  Tropical Storm  Tropical Storm
8      Depression      Depression
9  Tropical Storm  Tropical Storm


In [14]:
accuracy = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {accuracy:.2%}")



Test Accuracy: 91.98%


In [15]:
new_sample = pd.DataFrame({
    "Basin": ["ATLN"],
    "Longitude": [-70.0],
    "Latitude": [20.0],
    "DateTime": [2026110512],
    "PressureDrop": [80.0],
    "Pressure": [950.0]
})

prediction = model.predict(new_sample)

print("Predicted Cyclone Category:", prediction[0])

Predicted Cyclone Category: Hurricane
